# Quantisation

LLM inference is **memory-bound** during decode — the bottleneck is loading model
weights from HBM to compute units. Quantisation reduces the size of each weight,
meaning fewer bytes to load per forward pass = faster inference.

This notebook covers:
1. What quantisation is — float representations and precision
2. Why it works — weight distributions in LLMs
3. Quantisation methods — PTQ vs QAT, symmetric vs asymmetric
4. Granularity — per-tensor, per-channel, per-group
5. Practical formats — INT8, INT4, FP8, GPTQ, AWQ, GGUF
6. Quantising activations and KV cache
7. Quality vs speed tradeoffs

## 1. What quantisation is

Neural network weights are normally stored as 16-bit or 32-bit floating point numbers.
Quantisation maps these to lower-precision formats (8-bit, 4-bit, or even 2-bit),
reducing memory and bandwidth requirements.

### Floating point formats

```
Format     Bits   Range           Precision        Memory for 7B params
─────────────────────────────────────────────────────────────────────────
FP32       32     ±3.4×10³⁸       ~7 decimal       28 GB
FP16       16     ±65,504         ~3.3 decimal     14 GB
BF16       16     ±3.4×10³⁸       ~3.3 decimal     14 GB
FP8 (E4M3) 8      ±448            ~2 decimal        7 GB
INT8        8     -128 to 127     uniform steps     7 GB
INT4        4     -8 to 7         16 values only    3.5 GB
```

### The key insight

**Decode is memory-bound.** If we halve the weight size (16-bit → 8-bit), we halve
the bytes loaded from HBM per forward pass, roughly **doubling decode throughput**.

Going to 4-bit: another 2x reduction = 4x less memory traffic vs FP16.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# Demonstrate what happens when you quantise a weight tensor
torch.manual_seed(42)

# Simulate a typical LLM weight distribution (approximately normal)
weights_fp32 = torch.randn(1000) * 0.02  # typical init scale

def quantise_symmetric(tensor, bits):
    """Symmetric quantisation: map [-max, max] to [-2^(b-1), 2^(b-1)-1]"""
    qmax = 2 ** (bits - 1) - 1
    scale = tensor.abs().max() / qmax
    quantised = torch.round(tensor / scale).clamp(-qmax - 1, qmax)
    dequantised = quantised * scale
    return quantised.to(torch.int8 if bits == 8 else torch.int8), dequantised, scale

# Quantise to different bit widths
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

configs = [
    ("FP32 (original)", weights_fp32, 32),
    ("INT8 (256 levels)", *quantise_symmetric(weights_fp32, 8)[1::-1], 8),
    ("INT4 (16 levels)", *quantise_symmetric(weights_fp32, 4)[1::-1], 4),
    ("INT2 (4 levels)", *quantise_symmetric(weights_fp32, 2)[1::-1], 2),
]

for ax, (title, values, bits) in zip(axes.flat, configs):
    if isinstance(values, tuple):
        values = values[0]
    ax.hist(values.numpy(), bins=50, alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.set_title(f"{title}")
    ax.set_xlabel("Weight value")
    ax.set_ylabel("Count")
    unique_vals = len(torch.unique(values)) if bits < 32 else ">1M"
    ax.text(0.95, 0.95, f"{bits} bits\n{unique_vals} unique values",
            transform=ax.transAxes, ha='right', va='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle("Weight distribution at different quantisation levels", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Measure the quantisation error
errors = {}
for bits in [8, 4, 3, 2]:
    _, dequantised, scale = quantise_symmetric(weights_fp32, bits)
    mse = ((weights_fp32 - dequantised) ** 2).mean().item()
    max_err = (weights_fp32 - dequantised).abs().max().item()
    rel_err = ((weights_fp32 - dequantised).abs() / (weights_fp32.abs() + 1e-10)).mean().item()
    errors[bits] = {"mse": mse, "max_err": max_err, "rel_err": rel_err}

print(f"{'Bits':<6} {'MSE':<15} {'Max error':<15} {'Mean rel error':<15} {'Memory (7B)'}")
print("-" * 65)
for bits, e in errors.items():
    mem_gb = 7e9 * bits / 8 / 1e9
    print(f"{bits:<6} {e['mse']:<15.2e} {e['max_err']:<15.4f} {e['rel_err']*100:<15.2f}% {mem_gb:.1f} GB")

print(f"\nINT8: negligible error — essentially lossless for most tasks")
print(f"INT4: small error — quality depends on calibration method")
print(f"INT2: significant error — requires advanced techniques (QuIP#, etc.)")

## 2. Why quantisation works — weight distributions

LLM weights follow approximately **normal distributions** centered near zero.
Most values cluster in a narrow range, with rare outliers.

This is why quantisation works well:
- Most weights are small → well-represented by few bits
- Outliers are rare → can be handled specially
- The model is over-parameterised → robust to small perturbations

### The outlier problem
Some weights (and especially activations) have extreme outliers that blow up the
quantisation range, wasting precision on the common values:

```
Without outlier handling:    [-0.5 ......●●●●●●●●●●...... 0.5]  but one value at 10.0!
Scale set by outlier:        [-10 ............................●...... 10]  
                              Most values crammed into 2 bins  ↑ outlier gets its own bin

With outlier handling:       Keep outlier in FP16, quantise the rest with good precision
```

In [ ]:
# Demonstrate the outlier problem
torch.manual_seed(42)

# Normal weights with a few outliers (common in LLMs)
weights_normal = torch.randn(10000) * 0.02
weights_with_outliers = weights_normal.clone()
weights_with_outliers[0] = 2.0    # outlier!
weights_with_outliers[1] = -1.5   # outlier!

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Original
axes[0].hist(weights_with_outliers.numpy(), bins=100, alpha=0.7, color='steelblue')
axes[0].axvline(x=2.0, color='red', linestyle='--', label='Outliers')
axes[0].axvline(x=-1.5, color='red', linestyle='--')
axes[0].set_title("Original weights (with outliers)")
axes[0].legend()

# Naive INT4 quantisation (outliers ruin precision)
_, dequant_naive, _ = quantise_symmetric(weights_with_outliers, 4)
naive_error = ((weights_with_outliers - dequant_naive) ** 2).mean().item()
axes[1].hist(dequant_naive.numpy(), bins=100, alpha=0.7, color='orange')
axes[1].set_title(f"Naive INT4 (MSE: {naive_error:.2e})\nOutliers waste quantisation range")

# Clip outliers first, then quantise (simulating per-group or outlier-aware)
clip_val = weights_with_outliers.abs().quantile(0.99)
weights_clipped = weights_with_outliers.clamp(-clip_val, clip_val)
_, dequant_clipped, _ = quantise_symmetric(weights_clipped, 4)
clip_error = ((weights_normal - dequant_clipped[:len(weights_normal)]) ** 2).mean().item()
axes[2].hist(dequant_clipped.numpy(), bins=100, alpha=0.7, color='green')
axes[2].set_title(f"Outlier-aware INT4 (MSE: {clip_error:.2e})\nBetter precision for common values")

plt.tight_layout()
plt.show()
print(f"Naive quantisation MSE: {naive_error:.2e}")
print(f"Outlier-aware MSE:      {clip_error:.2e}  ({naive_error/clip_error:.1f}x better)")

## 3. Quantisation methods

### Post-Training Quantisation (PTQ)
Quantise a pre-trained model without retraining. Fast and cheap, but potentially
lower quality at very low bit widths.

### Quantisation-Aware Training (QAT)
Simulate quantisation during training so the model learns to be robust to it.
Higher quality but requires expensive retraining.

```
                Quality
                  ▲
                  │    ★ QAT (best quality, expensive)
                  │   
                  │  ● PTQ with calibration (GPTQ, AWQ)
                  │
                  │ ○ Naive PTQ (round-to-nearest)
                  │
                  └──────────────────────────────────► Cost
                    seconds      minutes      hours
```

### Symmetric vs asymmetric

| | Symmetric | Asymmetric |
|---|---|---|
| Zero point | Always 0 | Shifted (can represent offset distributions) |
| Formula | `q = round(x / scale)` | `q = round(x / scale) + zero_point` |
| Overhead | Lower (no zero_point math) | Higher (extra add per element) |
| Best for | Weights (centered at 0) | Activations (often positive, e.g. after ReLU) |

In [ ]:
# Symmetric vs asymmetric quantisation

def quantise_asymmetric(tensor, bits):
    """Asymmetric: map [min, max] to [0, 2^b - 1]"""
    qmin, qmax = 0, 2**bits - 1
    t_min, t_max = tensor.min(), tensor.max()
    scale = (t_max - t_min) / (qmax - qmin)
    zero_point = round((-t_min / scale).item())
    quantised = torch.round(tensor / scale + zero_point).clamp(qmin, qmax)
    dequantised = (quantised - zero_point) * scale
    return dequantised, scale, zero_point

# Simulate an asymmetric distribution (like activations after GeLU)
activations = torch.relu(torch.randn(5000)) * 0.5  # mostly positive

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Symmetric quantisation of positive-skewed data
_, dequant_sym, _ = quantise_symmetric(activations, 4)
sym_error = ((activations - dequant_sym) ** 2).mean().item()
axes[0].hist(dequant_sym.numpy(), bins=30, alpha=0.7, color='orange', label=f'MSE: {sym_error:.2e}')
axes[0].set_title("Symmetric INT4 on positive activations\n(wastes half the range on negatives)")
axes[0].legend()

# Asymmetric — uses full range
dequant_asym, _, _ = quantise_asymmetric(activations, 4)
asym_error = ((activations - dequant_asym) ** 2).mean().item()
axes[1].hist(dequant_asym.numpy(), bins=30, alpha=0.7, color='green', label=f'MSE: {asym_error:.2e}')
axes[1].set_title("Asymmetric INT4 on positive activations\n(full range used)")
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"Asymmetric is {sym_error/asym_error:.1f}x more precise for skewed distributions.")
print(f"For weights (centered at 0), symmetric is fine and faster.")

## 4. Granularity — where to compute scale factors

The scale factor determines how floating point values map to quantised integers.
Finer granularity = better precision but more overhead.

```
Per-tensor:   1 scale for entire weight matrix
              [████████████████████████████]  → 1 scale
              Cheapest, but one outlier ruins all values

Per-channel:  1 scale per output row/column
              [████] → scale₁
              [████] → scale₂  
              [████] → scale₃
              Better — outliers only affect their row

Per-group:    1 scale per group of G elements (e.g. G=128)
              [██|██|██|██]  → scale₁, scale₂, scale₃, scale₄
              Best precision, most overhead
              Standard for INT4 (GPTQ uses g=128)
```

### Memory overhead of scale factors
For a `[4096, 4096]` weight matrix quantised to INT4 with group_size=128:
- Weights: 4096 × 4096 × 0.5 bytes = 8 MB
- Scales: 4096 × (4096/128) × 2 bytes = 256 KB (FP16 scales)
- Overhead: ~3% — negligible

In [ ]:
# Demonstrate per-group quantisation
torch.manual_seed(42)

# Create a weight matrix where different columns have different magnitudes
# (common in LLMs — some features have larger weights than others)
weight_matrix = torch.randn(256, 256)
# Make some columns have 10x larger magnitudes
weight_matrix[:, ::16] *= 10

def quantise_per_group(tensor, bits, group_size):
    """Quantise with a separate scale per group of elements."""
    flat = tensor.reshape(-1)
    n_groups = len(flat) // group_size
    grouped = flat[:n_groups * group_size].reshape(n_groups, group_size)
    
    qmax = 2 ** (bits - 1) - 1
    scales = grouped.abs().max(dim=1, keepdim=True).values / qmax
    quantised = torch.round(grouped / scales).clamp(-qmax - 1, qmax)
    dequantised = (quantised * scales).reshape(-1)
    
    # Handle remainder
    result = flat.clone()
    result[:len(dequantised)] = dequantised
    return result.reshape(tensor.shape)

# Compare granularities
group_sizes = [256*256, 256, 128, 32]  # per-tensor, per-row, group=128, group=32
labels = ["Per-tensor\n(1 scale)", "Per-channel\n(256 scales)", "Group=128\n(512 scales)", "Group=32\n(2048 scales)"]
errors_by_group = []

for gs in group_sizes:
    dequant = quantise_per_group(weight_matrix, bits=4, group_size=gs)
    mse = ((weight_matrix - dequant) ** 2).mean().item()
    errors_by_group.append(mse)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(labels, errors_by_group, color=['red', 'orange', 'green', 'darkgreen'], alpha=0.7)
ax.set_ylabel("Mean Squared Error")
ax.set_title("INT4 quantisation error by granularity (matrix with varying column magnitudes)")
ax.grid(True, alpha=0.3, axis='y')

for bar, err in zip(bars, errors_by_group):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{err:.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print(f"Per-group (128) reduces error by {errors_by_group[0]/errors_by_group[2]:.0f}x vs per-tensor.")
print(f"This is why GPTQ/AWQ default to group_size=128.")

## 5. Practical quantisation formats

### Weight-only quantisation (most common for serving)

| Method | Bits | Calibration | Quality | Speed | Notes |
|--------|------|-------------|---------|-------|-------|
| **GPTQ** | 4 (3,8) | ~100 samples, layer-by-layer | Good | Medium | OBS-based; per-group; most widely supported |
| **AWQ** | 4 | Activation-aware | Better | Fast | Protects salient weights; better for edge |
| **GGUF/GGML** | 2–8 | Simple RTN or importance | Varies | CPU-optimised | llama.cpp ecosystem; mixed precision per layer |
| **SqueezeLLM** | 3–4 | Sensitivity-based | Good | Medium | Non-uniform quantisation; sparse + dense |
| **QuIP#** | 2–4 | Incoherence processing | State-of-art | Slow (decode) | Lattice-based; best quality at 2-bit |

### Weight + activation quantisation

| Method | Bits (W/A) | Notes |
|--------|-----------|-------|
| **SmoothQuant** | W8A8 | Migrates difficulty from activations to weights |
| **FP8** | W8A8 | Native on H100/MI300; drop-in for FP16 |
| **INT8** | W8A8 | Tensor cores on A100+; ~2x throughput |

### How GPTQ works (simplified)

```
For each layer:
  1. Run ~128 calibration samples through the model
  2. For each column of the weight matrix:
     a. Quantise it to INT4
     b. Measure the error introduced
     c. Distribute that error to remaining (unquantised) columns
        (Optimal Brain Surgeon-style compensation)
  3. Result: each column's quantisation error is partially compensated
     by adjusting subsequent columns
```

### How AWQ works (simplified)

```
Key insight: not all weights are equally important.
Weights connected to large activations matter more.

  1. Run calibration data, observe activation magnitudes
  2. Identify "salient" weight channels (connected to large activations)
  3. Scale salient channels UP before quantisation (protecting them)
  4. Scale corresponding activations DOWN to compensate
  5. Net effect: salient weights get better quantisation precision
```

In [ ]:
# Simulate AWQ's insight: protecting salient weights
torch.manual_seed(42)

# Weight matrix and activations
W = torch.randn(64, 64) * 0.02
X = torch.randn(100, 64)  # calibration activations

# Some channels have much larger activations (salient)
X[:, ::8] *= 10  # every 8th channel is 10x more active

# Ground truth output
Y_true = X @ W.T

# Naive quantisation: quantise all weights uniformly
_, W_naive_deq, _ = quantise_symmetric(W.flatten(), 4)
W_naive = W_naive_deq.reshape(W.shape)
Y_naive = X @ W_naive.T
naive_output_error = ((Y_true - Y_naive) ** 2).mean().item()

# AWQ-style: scale salient channels to protect them
activation_magnitude = X.abs().mean(dim=0)  # per-channel activation magnitude
# Scaling factor: protect channels proportional to activation magnitude
s = (activation_magnitude / activation_magnitude.mean()).clamp(min=0.5, max=5.0)

# Scale weights up (protect) and activations down (compensate)
W_scaled = W * s.unsqueeze(0)  # scale weight columns
_, W_awq_deq, _ = quantise_symmetric(W_scaled.flatten(), 4)
W_awq = W_awq_deq.reshape(W.shape)
W_awq_unscaled = W_awq / s.unsqueeze(0)  # undo scaling after quantisation
Y_awq = X @ W_awq_unscaled.T
awq_output_error = ((Y_true - Y_awq) ** 2).mean().item()

print(f"Output error comparison (INT4):")
print(f"  Naive quantisation:  MSE = {naive_output_error:.6f}")
print(f"  AWQ-style:           MSE = {awq_output_error:.6f}")
print(f"  Improvement:         {naive_output_error/awq_output_error:.1f}x better")
print(f"\nAWQ protects the channels that matter most to the output.")
print(f"Same 4-bit budget, but error-weighted toward important features.")

## 6. Quantising the KV cache

Beyond model weights, the **KV cache** is the other major memory consumer during inference.
For long sequences, it can exceed the model size itself.

### KV cache memory (full attention model, no quantisation)

```
Per token: 2 × num_layers × num_kv_heads × head_dim × bytes_per_element

LLaMA-70B example:
  = 2 × 80 layers × 8 KV heads × 128 head_dim × 2 bytes (FP16)
  = 327 KB per token
  
  At 128K context: 327 KB × 131,072 = 42 GB just for KV cache!
```

### KV cache quantisation

| Format | Memory reduction | Quality impact | Notes |
|--------|-----------------|---------------|-------|
| FP16 → FP8 | 2x | Negligible | Supported on H100 natively |
| FP16 → INT4 | 4x | Small (<1% perplexity) | Per-token or per-head quantisation |
| FP16 → INT2 | 8x | Noticeable | Research stage; requires careful calibration |

**Key difference from weight quantisation**: KV cache values are computed at runtime
(not known ahead of time), so we can't use calibration-based methods like GPTQ.
Instead, we quantise on-the-fly as values enter the cache.

In [ ]:
# KV cache memory calculator

def kv_cache_memory(num_layers, num_kv_heads, head_dim, seq_len, 
                    bytes_per_element=2, batch_size=1):
    """Calculate KV cache memory in GB."""
    # 2 for K and V
    total_bytes = 2 * num_layers * num_kv_heads * head_dim * seq_len * bytes_per_element * batch_size
    return total_bytes / 1e9

# Model configs
models = {
    "Qwen3.5-2B": {"layers": 24, "kv_heads": 2, "head_dim": 256},  # only 6 full-attn layers
    "LLaMA-3-8B": {"layers": 32, "kv_heads": 8, "head_dim": 128},
    "LLaMA-3-70B": {"layers": 80, "kv_heads": 8, "head_dim": 128},
    "GPT-4 class (est)": {"layers": 120, "kv_heads": 16, "head_dim": 128},
}

seq_lengths = [1024, 4096, 32768, 131072]
quantisation_formats = [
    ("FP16", 2),
    ("FP8", 1),
    ("INT4", 0.5),
]

print(f"KV Cache Memory (batch=1)")
print("=" * 80)

for model_name, cfg in models.items():
    print(f"\n{model_name} ({cfg['layers']}L, {cfg['kv_heads']}KV, d={cfg['head_dim']})")
    print(f"{'Seq len':<10}", end="")
    for fmt, _ in quantisation_formats:
        print(f"{fmt:<12}", end="")
    print()
    print("-" * 46)
    
    for seq_len in seq_lengths:
        print(f"{seq_len:<10}", end="")
        for fmt, bpe in quantisation_formats:
            mem = kv_cache_memory(cfg["layers"], cfg["kv_heads"], cfg["head_dim"], seq_len, bpe)
            print(f"{mem:<12.2f}", end="")
        print(" GB")

print(f"\n{'─'*80}")
print(f"At 128K context, LLaMA-70B KV cache in FP16 = "
      f"{kv_cache_memory(80, 8, 128, 131072, 2):.1f} GB")
print(f"With INT4 quantisation: {kv_cache_memory(80, 8, 128, 131072, 0.5):.1f} GB (4x smaller)")
print(f"\nThis determines how many concurrent requests can fit in GPU memory.")

## 7. Quality vs speed tradeoffs

### The Pareto frontier

```
Quality (perplexity)
     ▲
     │  ★ FP16 (baseline)
     │  ● FP8 (negligible loss, 2x speed)
     │  ● INT8 (negligible loss, 2x speed)  
     │
     │    ● GPTQ-4bit (small loss, 4x speed)
     │    ● AWQ-4bit (smaller loss, 4x speed)
     │
     │       ● GPTQ-3bit (moderate loss)
     │
     │          ○ INT2 (significant loss, 8x speed)
     │
     └─────────────────────────────────────────────► Speed / Memory
```

### Rules of thumb

| Scenario | Recommendation |
|----------|---------------|
| Maximum quality, GPU memory allows | FP16 / BF16 |
| Production serving (quality + speed) | INT8 or FP8 weights |
| Memory-constrained GPU | INT4 (GPTQ or AWQ, group=128) |
| CPU / edge deployment | INT4 GGUF (llama.cpp) |
| Long context serving | INT4 weights + FP8 KV cache |
| Research / bleeding edge | INT2-3 (QuIP#, significant quality loss) |

### The practical sweet spot

**INT4 weights (AWQ/GPTQ) + FP8 KV cache** is the current sweet spot for most
production deployments:
- 4x weight memory reduction → fits larger models on fewer GPUs
- 2x KV cache reduction → serves more concurrent requests
- <1% quality degradation on most benchmarks
- Supported by vLLM, TensorRT-LLM, SGLang out of the box

In [ ]:
# Final comparison: model size and theoretical throughput at different precisions

param_count_B = 70  # 70B parameter model

configs = [
    ("FP32", 4, 1.0),
    ("FP16/BF16", 2, 1.0),
    ("FP8", 1, 0.99),
    ("INT8 (SmoothQuant)", 1, 0.99),
    ("INT4 (GPTQ g=128)", 0.55, 0.97),  # 0.5 + scale overhead
    ("INT4 (AWQ)", 0.55, 0.98),
    ("INT3 (GPTQ)", 0.42, 0.93),
    ("INT2 (QuIP#)", 0.3, 0.88),
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

names = [c[0] for c in configs]
sizes = [param_count_B * c[1] for c in configs]
quality = [c[2] * 100 for c in configs]

colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(configs)))

# Model size
bars = ax1.barh(names, sizes, color=colors)
ax1.set_xlabel("Model size (GB)")
ax1.set_title(f"{param_count_B}B parameter model — size at each precision")
ax1.axvline(x=80, color='red', linestyle='--', alpha=0.5, label='A100 80GB')
ax1.axvline(x=24, color='orange', linestyle='--', alpha=0.5, label='RTX 4090 24GB')
ax1.legend()
for bar, size in zip(bars, sizes):
    ax1.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
             f'{size:.0f} GB', va='center', fontsize=9)

# Quality
ax2.barh(names, quality, color=colors)
ax2.set_xlabel("Relative quality (% of FP16)")
ax2.set_title("Approximate quality retention")
ax2.set_xlim(80, 101)
ax2.axvline(x=99, color='green', linestyle='--', alpha=0.3, label='99% (negligible loss)')
ax2.axvline(x=95, color='orange', linestyle='--', alpha=0.3, label='95% (noticeable)')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"\n70B model fitting on a single GPU:")
print(f"  FP16: needs 140 GB → 2× A100-80GB minimum")
print(f"  INT8: needs 70 GB  → fits on 1× A100-80GB")
print(f"  INT4: needs ~38 GB → fits on 1× A100-40GB or 2× RTX 4090")
print(f"\nQuantisation directly determines what hardware you need.")